In [ ]:
!pip install kaggle
# Ensure you upload your kaggle.json API token to Colab first
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download and extract
!kaggle datasets download -d preatcher/standard-ocr-dataset
!unzip -q standard-ocr-dataset.zip -d dataset/

In [2]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Define tensor transformations and normalization
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((32, 32)), # Standardizing input dimensions for the CNN
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# 2. Load the dataset (Adjust path based on the exact unzip output, usually 'data/training_data')
train_data = datasets.ImageFolder(root='dataset/data/training_data', transform=transform)

# 3. Optimize the dataloader to keep the T4 GPU saturated
train_loader = DataLoader(train_data, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)

print(f"Loaded {len(train_data)} images across {len(train_data.classes)} classes.")

Loaded 20628 images across 36 classes.


In [3]:
import torch
import torch.nn as nn

class NanoOCR(nn.Module):
    def __init__(self, num_classes: int = 36):
        super(NanoOCR, self).__init__()

        # Block 1: Input (1, 32, 32) -> Output (32, 16, 16)
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Block 2: Input (32, 16, 16) -> Output (64, 8, 8)
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Block 3: Input (64, 8, 8) -> Output (128, 4, 4)
        self.conv3 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Fully Connected Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        logits = self.classifier(x)
        return logits

In [4]:
# Select GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize model
model = NanoOCR(num_classes=36).to(device)

# Loss function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Print parameter count to verify it remains ultra-lightweight
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"NanoOCR initialized on: {device}")
print(f"Total Trainable Parameters: {total_params:,}")

NanoOCR initialized on: cuda
Total Trainable Parameters: 626,916


In [5]:
import time
from torch.utils.data import random_split, DataLoader

# 1. Split the dataset (80% Train, 20% Validation)
train_size = int(0.8 * len(train_data))
val_size = len(train_data) - train_size
train_dataset, val_dataset = random_split(train_data, [train_size, val_size])

# 2. Re-create the DataLoaders with the new splits
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

# 3. The Training Loop
epochs = 5

for epoch in range(epochs):
    start_time = time.time()

    # --- TRAINING PHASE ---
    model.train()  # Sets the model to training mode (enables Dropout & BatchNorm)
    running_loss = 0.0
    correct_train, total_train = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()           # Clear old gradients
        outputs = model(images)         # Forward pass
        loss = criterion(outputs, labels) # Calculate error
        loss.backward()                 # Backward pass
        optimizer.step()                # Update weights

        # Track training accuracy
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_acc = 100 * correct_train / total_train
    avg_train_loss = running_loss / len(train_loader)

    # --- VALIDATION PHASE ---
    model.eval()   # Sets the model to evaluation mode (disables Dropout)
    val_loss = 0.0
    correct_val, total_val = 0, 0

    with torch.no_grad(): # Disable gradient calculation for speed & memory
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_acc = 100 * correct_val / total_val
    avg_val_loss = val_loss / len(val_loader)

    # --- PRINT EPOCH METRICS ---
    elapsed = time.time() - start_time
    print(f"Epoch [{epoch+1}/{epochs}] | Time: {elapsed:.0f}s")
    print(f"  Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {avg_val_loss:.4f} | Val Acc:   {val_acc:.2f}%\n")

Epoch [1/5] | Time: 12s
  Train Loss: 0.5894 | Train Acc: 84.48%
  Val Loss:   0.1387 | Val Acc:   96.03%

Epoch [2/5] | Time: 7s
  Train Loss: 0.1557 | Train Acc: 95.30%
  Val Loss:   0.1031 | Val Acc:   96.83%

Epoch [3/5] | Time: 9s
  Train Loss: 0.1134 | Train Acc: 96.22%
  Val Loss:   0.1012 | Val Acc:   96.44%

Epoch [4/5] | Time: 8s
  Train Loss: 0.0913 | Train Acc: 96.71%
  Val Loss:   0.1093 | Val Acc:   96.24%

Epoch [5/5] | Time: 8s
  Train Loss: 0.0763 | Train Acc: 97.22%
  Val Loss:   0.0830 | Val Acc:   97.46%



In [ ]:
!pip install onnxscript onnx

In [ ]:
import json
import torch

# 1. Save the class mappings to a JSON file
class_mapping = {i: c for i, c in enumerate(train_data.classes)}
with open("class_mapping.json", "w") as f:
    json.dump(class_mapping, f)

# 2. Export the model to ONNX format
model.eval() # MUST be in evaluation mode for export
dummy_input = torch.randn(1, 1, 32, 32, device=device) # (Batch, Channels, Height, Width)

torch.onnx.export(
    model,
    dummy_input,
    "nano_ocr.onnx",
    export_params=True,
    opset_version=14,        # Standard, stable ONNX opset
    do_constant_folding=True, # Optimizes constant mathematical operations
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"}, # Allows the API to process multiple characters at once
        "output": {0: "batch_size"}
    }
)

print("✅ Exported nano_ocr.onnx and class_mapping.json successfully!")

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
✅ Exported nano_ocr.onnx and class_mapping.json successfully!